In [1]:
!pip install -q fastapi uvicorn httpx

import os
import sys
import importlib
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import LinearRegression

np.random.seed(42)
n = 500

df = pd.DataFrame()
df["distance_km"]   = np.random.uniform(1, 15, n)
df["items"]         = np.random.randint(1, 10, n)
df["traffic_level"] = np.random.randint(1, 4, n)
df["delivery_minutes"] = (10 + 3 * df["distance_km"] + 1.5 * df["items"]
                          + 5 * df["traffic_level"] + np.random.normal(0, 3, n))

X = df[["distance_km", "items", "traffic_level"]]
y = df["delivery_minutes"]

model = LinearRegression()
model.fit(X, y)

os.makedirs("work", exist_ok=True)
joblib.dump(model, "work/model.joblib")
print("Model saved to work/model.joblib")

Model saved to work/model.joblib


In [2]:
%%writefile work/app.py
import os
import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, Field

# Load the saved model once, when the service starts
model = joblib.load(os.path.join(os.path.dirname(__file__), "model.joblib"))

app = FastAPI(title="Delivery Time Prediction Service")


# The shape of a valid request (ranges match the training data)
class Order(BaseModel):
    distance_km:   float = Field(gt=0, le=15)
    items:         int   = Field(ge=1, le=9)
    traffic_level: int   = Field(ge=1, le=3)


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/predict")
def predict(order: Order):
    # Convert the order into a one-row table, as the model expects
    row = pd.DataFrame([order.model_dump()])
    prediction = model.predict(row)[0]
    return {"predicted_minutes": round(float(prediction), 1)}

Writing work/app.py


In [4]:
from fastapi.testclient import TestClient

sys.path.insert(0, "work")
import app
importlib.reload(app)

client = TestClient(app.app)

response = client.get("/health")
print("Health  :", response.status_code, response.json())

order = {"distance_km": 8, "items": 3, "traffic_level": 2}
response = client.post("/predict", json=order)
print("Predict :", response.status_code, response.json())

# Step 4: an impossible traffic level
bad_order = {"distance_km": 8, "items": 3, "traffic_level": 9}
response = client.post("/predict", json=bad_order)
print("Status code:", response.status_code)
print("Problem    :", response.json()["detail"][0]["msg"])

Health  : 200 {'status': 'ok'}
Predict : 200 {'predicted_minutes': 48.4}
Status code: 422
Problem    : Input should be less than or equal to 3


In [5]:
import subprocess, time
from google.colab import output

server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app:app", "--port", "8000"],
    cwd="work",
)

time.sleep(8)
output.serve_kernel_port_as_iframe(8000, path="/docs", height=700)

<IPython.core.display.Javascript object>

In [6]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression

iris = load_iris()
X, y = iris.data, iris.target
print("Species :", iris.target_names)
print("Features:", iris.feature_names)

model = LogisticRegression(max_iter=1000)
model.fit(X, y)

os.makedirs("task", exist_ok=True)
joblib.dump(model, "task/iris_model.joblib")
print("Model saved to task/iris_model.joblib")

Species : ['setosa' 'versicolor' 'virginica']
Features: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Model saved to task/iris_model.joblib


In [7]:
%%writefile task/iris_app.py
import os
import joblib
from fastapi import FastAPI
from pydantic import BaseModel, Field
from sklearn.datasets import load_iris

# Load the saved model once, when the service starts
model = joblib.load(os.path.join(os.path.dirname(os.path.abspath(__file__)), "iris_model.joblib"))

# Class numbers 0, 1, 2 -> species names (setosa, versicolor, virginica)
SPECIES = load_iris().target_names

app = FastAPI(title="Iris Flower Classification Service")


# A valid request: every measurement must be > 0 and at most 10 cm
class Flower(BaseModel):
    sepal_length: float = Field(gt=0, le=10)
    sepal_width:  float = Field(gt=0, le=10)
    petal_length: float = Field(gt=0, le=10)
    petal_width:  float = Field(gt=0, le=10)


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/predict")
def predict(flower: Flower):
    # Same feature order as the training data
    features = [[flower.sepal_length, flower.sepal_width,
                 flower.petal_length, flower.petal_width]]
    class_number = int(model.predict(features)[0])
    return {"species": str(SPECIES[class_number])}

Writing task/iris_app.py


In [8]:
import sys, importlib
from fastapi.testclient import TestClient

sys.path.insert(0, "task")
import iris_app
importlib.reload(iris_app)

iris_client = TestClient(iris_app.app)

print("Health :", iris_client.get("/health").json())

flower = {"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}
response = iris_client.post("/predict", json=flower)
print("Predict:", response.status_code, response.json())

bad_flower = {"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": -1.0, "petal_width": 0.2}
response = iris_client.post("/predict", json=bad_flower)
print("Status code:", response.status_code)
print("Problem    :", response.json()["detail"][0]["msg"])

Health : {'status': 'ok'}
Predict: 200 {'species': 'setosa'}
Status code: 422
Problem    : Input should be greater than 0


In [9]:
iris_server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "iris_app:app", "--port", "8001"],
    cwd="task",
)

time.sleep(8)
output.serve_kernel_port_as_iframe(8001, path="/docs", height=800)

<IPython.core.display.Javascript object>